In [1]:
import numpy as np

# -----------------------------
# ОПИСАНИЕ СРЕДЫ
# -----------------------------

# В мире 5 состояний: 0, 1, 2, 3, 4
# Состояние 4 считаем "целью справа"
n_states = 5

# 2 возможных действия:
# 0 = идти влево
# 1 = идти вправо
n_actions = 2

# Q-таблица размера [число состояний x число действий]
# Q[s, a] = насколько выгодно сделать действие a в состоянии s
#
# Изначально все значения 0, потому что агент пока ничего не знает о мире
Q = np.zeros((n_states, n_actions))

# -----------------------------
# ГИПЕРПАРАМЕТРЫ ОБУЧЕНИЯ
# -----------------------------

# alpha — скорость обучения
# Насколько сильно новая информация меняет старую оценку
alpha = 0.1

# gamma — discount factor
# Насколько важны будущие награды
gamma = 0.9

# epsilon — вероятность исследования
# С вероятностью epsilon агент делает случайное действие,
# а не лучшее по текущей Q-таблице
epsilon = 0.2

In [2]:
# -----------------------------
# ФУНКЦИЯ ПЕРЕХОДА СРЕДЫ
# -----------------------------

def step(state, action):
    # Если action == 0, двигаемся влево
    if action == 0:
        # max(0, ...) не даёт выйти за левую границу
        next_state = max(0, state - 1)
    else:
        # Иначе action == 1, двигаемся вправо
        # min(n_states - 1, ...) не даёт выйти за правую границу
        next_state = min(n_states - 1, state + 1)

    # Награда 1, если пришли в состояние 4
    # Иначе награда 0
    reward = 1 if next_state == 4 else 0

    # Возвращаем:
    # - следующее состояние
    # - награду за переход
    return next_state, reward

In [5]:
# -----------------------------
# ЦИКЛ ОБУЧЕНИЯ
# -----------------------------

# Проходим 500 эпизодов
# Эпизод = одна попытка агента походить по среде
for episode in range(500):

    # В начале каждого эпизода агент стартует из состояния 0
    state = 0

    # Внутри эпизода разрешаем максимум 20 шагов
    for _ in range(20):

        # --------------------------------------
        # ВЫБОР ДЕЙСТВИЯ: exploration/exploitation
        # --------------------------------------

        # exploration:
        # с вероятностью epsilon выбираем случайное действие
        if np.random.rand() < epsilon:
            action = np.random.randint(2)
        else:
            # exploitation:
            # выбираем действие с максимальным текущим Q-значением
            action = np.argmax(Q[state])

        # Делаем шаг в среде:
        # из состояния state с действием action
        next_state, reward = step(state, action)

        # --------------------------------------
        # Q-learning update
        # --------------------------------------
        #
        # Формула:
        #
        # Q(s_t, a_t) <- Q(s_t, a_t) +
        #                alpha * (r_{t+1} + gamma * max_a' Q(s_{t+1}, a') - Q(s_t, a_t))
        #
        # Где:
        # state      = s_t
        # action     = a_t
        # next_state = s_{t+1}
        # reward     = r_{t+1}
        #
        # Внутри скобок:
        # reward + gamma * np.max(Q[next_state])
        #
        # это target, то есть новая "цель" для Q-value:
        # текущая награда + лучшая будущая ценность
        #
        # После этого из target вычитается старая оценка:
        # ... - Q[state, action]
        #
        # Это TD-error:
        # насколько старая оценка ошибалась
        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )

        # Переходим в новое состояние
        state = next_state

# После обучения печатаем Q-таблицу
print(Q)

[[ 6.56099589  7.29      ]
 [ 6.56099942  8.1       ]
 [ 7.28999993  9.        ]
 [ 8.1        10.        ]
 [ 9.         10.        ]]
